In [15]:
import pandas as pd
import numpy as np
marketing_campaign_df=pd.read_csv(r"D:\DataScience\GUVI\Local\Marketing_campaign_analysis\marketing_campaign_data.csv")
marketing_campaign_df

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Response,Complain,Country
0,342199,1985,Graduation,Together,59011.7,1,0,2012-11-17,3,0,...,3,4,0,0,0,0,0,0,0,Spain
1,8075450,1975,Master,Single,1730.0,1,1,2013-04-10,96,0,...,2,3,0,0,0,0,0,0,0,Spain
2,13664263,1978,Graduation,Married,98584.6,0,0,2014-01-11,99,920,...,6,3,0,0,0,0,0,0,0,Australia
3,16164787,1976,Graduation,Married,74031.5,1,0,2014-06-18,47,265,...,11,4,0,0,0,0,0,0,0,Spain
4,15815139,1981,Graduation,Divorced,52784.2,1,1,2014-05-20,0,30,...,3,6,0,0,0,0,0,0,0,Canada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55995,2485439,1985,Graduation,Married,32533.4,0,0,2013-04-05,99,8,...,6,8,0,0,0,0,0,0,0,Australia
55996,9202496,1992,Basic,Single,70496.3,0,1,2013-01-31,9,835,...,8,4,0,0,0,0,0,0,0,Germany
55997,14677746,1950,Graduation,Together,21072.0,1,0,2013-06-25,24,0,...,0,3,0,0,0,0,0,1,0,Saudi Arabia
55998,3843719,1976,Graduation,Together,64218.1,0,0,2014-05-12,81,8,...,1,8,0,0,0,0,0,0,0,Saudi Arabia


In [12]:
marketing_campaign_df['Dt_Customer']=pd.to_datetime(marketing_campaign_df['Dt_Customer'])

In [23]:


clean_df = pd.read_pickle(r"D:\DataScience\GUVI\Local\Marketing_campaign_analysis\cleaned_data.pkl")
clean_df.info()

<class 'pandas.DataFrame'>
Index: 55986 entries, 0 to 55999
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   55986 non-null  int64         
 1   Education            55986 non-null  str           
 2   Marital_Status       55986 non-null  str           
 3   Income               55986 non-null  float64       
 4   Kidhome              55986 non-null  int64         
 5   Teenhome             55986 non-null  int64         
 6   Dt_Customer          55986 non-null  datetime64[us]
 7   Recency              55986 non-null  int64         
 8   MntWines             55986 non-null  int64         
 9   MntFruits            55986 non-null  int64         
 10  MntMeatProducts      55986 non-null  int64         
 11  MntFishProducts      55986 non-null  int64         
 12  MntSweetProducts     55986 non-null  int64         
 13  MntGoldProds         55986 non-null  int64     

## Create a SQL table

In [ ]:
#to make a connection
import mysql.connector 

connection = mysql.connector.connect(
    host = "Localhost",
    user = "root",
    password = "12345"
)
cursor =  connection.cursor()
cursor

In [5]:
query= "create database if not exists Market_analysis"
cursor.execute(query)
connection.commit()

In [6]:
query = "use Market_analysis"
cursor.execute(query)

In [18]:
query = """create table if not exists marketing_campaigns_data(ID int primary key, Year_Birth int, Education varchar(60), Marital_Status varchar(60), Income float, Kidhome int, Teenhome int, Dt_Customer DATE, Recency int, MntWines int, MntFruits int, MntMeatProducts int, MntFishProducts int, MntSweetProducts int, MntGoldProds int, NumDealsPurchases int, NumWebPurchases int, NumCatalogPurchases int, NumStorePurchases int, NumWebVisitsMonth int, AcceptedCmp3 int,  AcceptedCmp4 int, AcceptedCmp5 int, AcceptedCmp1 int, AcceptedCmp2 int, Response int, Complain int, Country varchar(60))"""
cursor.execute(query)
connection.commit()

In [20]:
columns = ",".join(marketing_campaign_df.columns)
placeholders = ",".join(["%s"]*len(marketing_campaign_df.columns))
query = f"insert into marketing_campaigns_data({columns}) values({placeholders})"
tuple_data = []

for idx in marketing_campaign_df.index:
    row = list(marketing_campaign_df.loc[idx].values)

    for j in range(len(row)):
        if isinstance(row[j], (np.integer, np.int64)):
            row[j] = int(row[j])
        elif isinstance(row[j], pd.Timestamp):
            row[j] = row[j].strftime('%Y-%m-%d')
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## Derived Table

In [33]:
query="create table if not exists derived_table(ID int primary key, Education varchar(60), Marital_Status varchar(60), Income float, Recency int, NumWebVisitsMonth int, AcceptedCmp3 int,  AcceptedCmp4 int, AcceptedCmp5 int, AcceptedCmp1 int, AcceptedCmp2 int, Response int, Complain int, Country varchar(60), Age int, Children int, Total_Spend int, Total_Purchases int, Family_Size int, Customer_since int)"
cursor.execute(query)
connection.commit()

In [28]:
selected_cols = ['ID', 'Education', 'Marital_Status', 'Income', 'Recency', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Response', 'Complain', 'Country', 'Age' , 'Children', 'Total_Spend', 'Total_Purchases', 'Family_Size', 'Customer_since']

selected_df = clean_df[selected_cols]

In [34]:
columns = ",".join(df.columns)
placeholders = ",".join(["%s"]*len(selected_df.columns))
query = f"insert into derived_table({columns}) values({placeholders})"
tuple_data = []

for idx in selected_df.index:
    row = list(selected_df.loc[idx].values)

    for j in range(len(row)):
        if isinstance(row[j], (np.integer, np.int64)):
            row[j] = int(row[j])
        
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

## Derived Segmentation Table

In [35]:
seg_df=pd.read_pickle(r"D:\DataScience\GUVI\Local\Marketing_campaign_analysis\incl_segmentation.pkl")

In [37]:
query="Create table if not exists derived_segmentation_table(ID INT PRIMARY KEY, High_Income VARCHAR(10), Young_Customer VARCHAR(10), Campaign_Responder VARCHAR(10), High_Web_Engagement VARCHAR(10), Family_Customer VARCHAR(10), High_Spender VARCHAR(20))"
cursor.execute(query)
connection.commit()

In [38]:
seg_selected_cols = ['ID', 'High_Income', 'Young_Customer', 'Campaign_Responder', 'High_Web_Engagement', 'Family_Customer', 'High_Spender']

seg_selected_df = seg_df[seg_selected_cols]

In [41]:
columns = ",".join(seg_selected_df.columns)
placeholders = ",".join(["%s"]*len(seg_selected_df.columns))
query = f"insert ignore into derived_segmentation_table({columns}) values({placeholders})"
tuple_data = []
for i in seg_selected_df.index:
    row = list(seg_selected_df.loc[i].values)
    row[0]=int(row[0])
    row = tuple(row)
    tuple_data.append(row)
   
cursor.executemany(query, tuple_data)
connection.commit()

In [ ]:
cursor.close()